In [5]:
!pip install scikit-learn

In [6]:
!pip install torch torchvision torchaudio

  Using cached torch-2.12.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchaudio-2.11.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached cuda_bindings-13.3.1-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.29.7-py3-none-manylinux_2_18_x86_64.wh

In [18]:
"""
Problema 1
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
import os  # Para manejo y verificación de carpetas virtuales o locales
from IPython.display import display, Image  # Corrección explícita para renderizar en pantalla

warnings.filterwarnings("ignore")

# ── CONFIGURACIÓN DE RUTAS Y ALMACENAMIENTO ──────────────────────────────────
output_dir = "outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# PARÁMETROS GLOBALES
np.random.seed(88)
torch.manual_seed(88)

N_SAMPLES  = 3000       # número de señales
NT         = 1000       # puntos temporales por señal
T_END      = 10.0       # intervalo de tiempo [0, T_END]
SIGMA_BASE = 0.02       # ruido base para 1b/1c
M          = 1.0        # masa (fija)

t_eval = np.linspace(0, T_END, NT)


# 1B) GENERACIÓN DE DATOS

def solve_oscillator(gamma, k, t_eval, sigma=SIGMA_BASE):
    """Resuelve la EDO del oscilador usando RK45 y añade ruido gaussiano."""
    def ode(t, y):
        return [y[1], -gamma*y[1] - k*y[0]]

    sol = solve_ivp(ode, [0, t_eval[-1]], [1.0, 0.0],
                    t_eval=t_eval, method="RK45", rtol=1e-8, atol=1e-10)
    x = sol.y[0]
    if sigma > 0:
        x = x + np.random.normal(0, sigma, size=x.shape)
    return x


def generate_dataset(n_samples=N_SAMPLES, sigma=SIGMA_BASE, seed=88):
    """Generamos N señales con parámetros uniformes y ruido gaussiano."""
    rng = np.random.default_rng(seed)
    gammas = rng.uniform(0.05, 1.0, n_samples)
    ks     = rng.uniform(1.0,  5.0, n_samples)

    X = np.zeros((n_samples, NT))
    for i in range(n_samples):
        X[i] = solve_oscillator(gammas[i], ks[i], t_eval, sigma=sigma)

    theta = np.column_stack([gammas, ks])
    return X, theta


# Generar dataset base (sigma = 0.02)
X_data, theta_data = generate_dataset(N_SAMPLES, sigma=SIGMA_BASE)

# Graficamos las señales representativas 
fig1, axes1 = plt.subplots(2, 3, figsize=(13, 7))
examples = [
    (0.1, 1.0,  "$\\gamma$ bajo, k bajo\n(amortiguamiento lento, baja freq.)"),
    (0.1, 5.0,  "$\\gamma$ bajo, k alto\n(amortiguamiento lento, alta freq.)"),
    (0.5, 1.0,  "$\\gamma$ medio, k bajo\n(amortiguamiento medio, baja freq.)"),
    (0.5, 5.0,  "$\\gamma$ medio, k alto\n(amortiguamiento medio, alta freq.)"),
    (1.0, 1.0,  "$\\gamma$ alto, k bajo\n(amortiguamiento rápido, baja freq.)"),
    (1.0, 5.0,  "$\\gamma$ alto, k alto\n(amortiguamiento rápido, alta freq.)"),
]
for ax, (g, k, title) in zip(axes1.flat, examples):
    x_clean = solve_oscillator(g, k, t_eval, sigma=0.0)
    x_noisy = solve_oscillator(g, k, t_eval, sigma=SIGMA_BASE)
    ax.plot(t_eval, x_clean, "b-",  lw=1.5, label="sin ruido")
    ax.plot(t_eval, x_noisy, "r--", lw=0.8, alpha=0.7, label="con ruido")
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("t")
    ax.set_ylabel("x(t)")
    ax.legend(fontsize=7)

axes1[0, 0].set_title(examples[0][2] + "\n$\\rightarrow$ mayor k $\\rightarrow$ mayor $\\omega_d$", fontsize=7)
fig1.suptitle("1b) Señales del oscilador amortiguado\n"
             r"$k$ controla la frecuencia ($\omega_d$), "
             r"$\gamma$ controla el decaimiento ($e^{-\gamma t/2}$)", fontsize=10)
plt.tight_layout()

# Guardado estratégico y despliegue del Gráfico 1B
path1 = os.path.join(output_dir, "1b_senales_oscilador.png")
plt.savefig(path1, dpi=150)
display(fig1)
plt.close(fig1)


# 1C) SEPARACIÓN 80/20 Y ENTRENAMIENTO DE MODELOS

X_train, X_test, y_train, y_test = train_test_split(
    X_data, theta_data, test_size=0.2, random_state=88
)

# ── MODELO 1: RANDOM FOREST ─────────────────────────────────────────────────
print("\nEntrenando RandomForest...")
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=88
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rmse_rf_gamma = np.sqrt(mean_squared_error(y_test[:, 0], y_pred_rf[:, 0]))
rmse_rf_k     = np.sqrt(mean_squared_error(y_test[:, 1], y_pred_rf[:, 1]))
print(f"  RF  → RMSE_\\gamma = {rmse_rf_gamma:.4f},  RMSE_k = {rmse_rf_k:.4f}")


# ── MODELO 2: MLP PYTORCH ────────────────────────────────────────────────────

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 2),   # Predice (\gamma, k)
        )

    def forward(self, x):
        return self.net(x)


def train_mlp(X_tr, y_tr, X_val, y_val, epochs=80, batch_size=128, lr=1e-3):
    device = "cpu"
    Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
    ytr_t = torch.tensor(y_tr, dtype=torch.float32)
    Xval_t = torch.tensor(X_val, dtype=torch.float32)
    yval_t = torch.tensor(y_val, dtype=torch.float32)

    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)

    model = MLPRegressor(input_dim=X_tr.shape[1]).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=25, gamma=0.5)
    loss_fn = nn.MSELoss()

    train_losses, val_losses = [], []
    for ep in range(epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(xb)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xval_t), yval_t).item()
        train_losses.append(ep_loss / len(X_tr))
        val_losses.append(val_loss)

    model.eval()
    with torch.no_grad():
        y_pred = model(Xval_t).numpy()

    return model, y_pred, train_losses, val_losses


_, y_pred_mlp, tr_loss, val_loss = train_mlp(X_train, y_train, X_test, y_test)

rmse_mlp_gamma = np.sqrt(mean_squared_error(y_test[:, 0], y_pred_mlp[:, 0]))
rmse_mlp_k     = np.sqrt(mean_squared_error(y_test[:, 1], y_pred_mlp[:, 1]))
print(f"  MLP → RMSE_\\gamma = {rmse_mlp_gamma:.4f},  RMSE_k = {rmse_mlp_k:.4f}")

# Graficamos las predicciones vs valores reales 
fig2, axes2 = plt.subplots(2, 2, figsize=(11, 9))

for col, (preds, name) in enumerate([(y_pred_rf, "Random Forest"), (y_pred_mlp, "MLP PyTorch")]):
    for row, (param_idx, pname, rmse) in enumerate([
        (0, "\\gamma", [rmse_rf_gamma, rmse_mlp_gamma][col]),
        (1, "k", [rmse_rf_k,      rmse_mlp_k    ][col]),
    ]):
        ax = axes2[row, col]
        ax.scatter(y_test[:, param_idx], preds[:, param_idx],
                   alpha=0.3, s=8, color=["steelblue","tomato"][col])
        lo = y_test[:, param_idx].min()
        hi = y_test[:, param_idx].max()
        ax.plot([lo, hi], [lo, hi], "k--", lw=1.5, label="ideal")
        ax.set_xlabel(f"${pname}$ real")
        ax.set_ylabel(f"${pname}$ predicho")
        ax.set_title(f"{name} – ${pname}$\nRMSE = {rmse:.4f}")
        ax.legend(fontsize=8)

fig2.suptitle("1c) Predicciones vs. valores reales ($\\sigma$ = 0.02)", fontsize=12)
plt.tight_layout()

# Guardado estratégico y despliegue del Gráfico 1C (Predicciones)
path2 = os.path.join(output_dir, "1c_predicciones_vs_reales.png")
plt.savefig(path2, dpi=150)
display(fig2)
plt.close(fig2)


# Graficamos las curvas de entrenamiento de la MLP 
fig3, ax3 = plt.subplots(figsize=(7, 4))
ax3.semilogy(tr_loss,  label="Train MSE")
ax3.semilogy(val_loss, label="Val  MSE")
ax3.set_xlabel("Época")
ax3.set_ylabel("MSE (escala log)")
ax3.set_title("1c) Curvas de pérdida MLP")
ax3.legend()
plt.tight_layout()

# Guardado estratégico y despliegue del Gráfico 1C (Curva de pérdidas)
path3 = os.path.join(output_dir, "1c_curvas_perdida_mlp.png")
plt.savefig(path3, dpi=150)
display(fig3)
plt.close(fig3)


# 1D) ESTUDIO DEL EFECTO DEL RUIDO

sigmas = [0.0, 0.01, 0.02, 0.05, 0.10]
results = []   # lista de dicts

for sigma in sigmas:
    print(f"\n── \\sigma = {sigma} ──────────────────────────────────")
    Xs, ys = generate_dataset(N_SAMPLES, sigma=sigma, seed=0)
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=88)
        
    # Random Forest
    rf_s = RandomForestRegressor(n_estimators=150, max_depth=18,
                                  n_jobs=-1, random_state=88)
    rf_s.fit(Xtr, ytr)
    prf = rf_s.predict(Xte)
    r_rf_g = np.sqrt(mean_squared_error(yte[:, 0], prf[:, 0]))
    r_rf_k = np.sqrt(mean_squared_error(yte[:, 1], prf[:, 1]))
    print(f"  RF  → RMSE_\\gamma={r_rf_g:.4f}, RMSE_k={r_rf_k:.4f}")

    # MLP
    _, pmlp, _, _ = train_mlp(Xtr, ytr, Xte, yte, epochs=60)
    r_mlp_g = np.sqrt(mean_squared_error(yte[:, 0], pmlp[:, 0]))
    r_mlp_k = np.sqrt(mean_squared_error(yte[:, 1], pmlp[:, 1]))
    print(f"  MLP → RMSE_\\gamma={r_mlp_g:.4f}, RMSE_k={r_mlp_k:.4f}")

    results.append(dict(sigma=sigma,
                        rf_g=r_rf_g, rf_k=r_rf_k,
                        mlp_g=r_mlp_g, mlp_k=r_mlp_k))

# Graficamos el RMSE vs nivel de ruido
sig_vals = [r["sigma"] for r in results]
fig4, axes4 = plt.subplots(1, 2, figsize=(12, 5))

for ax, param, key_rf, key_mlp, color_rf, color_mlp in [
    (axes4[0], "\\gamma", "rf_g", "mlp_g", "steelblue", "tomato"),
    (axes4[1], "k", "rf_k", "mlp_k", "navy",      "firebrick"),
]:
    ax.plot(sig_vals, [r[key_rf]  for r in results], "o-", color=color_rf, lw=2, ms=7, label="Random Forest")
    ax.plot(sig_vals, [r[key_mlp] for r in results], "s--", color=color_mlp, lw=2, ms=7, label="MLP PyTorch")
    ax.set_xlabel("$\\sigma$ (nivel de ruido)", fontsize=11)
    ax.set_ylabel(f"RMSE_{param}", fontsize=11)
    ax.set_title(f"RMSE_{param} vs. $\\sigma$", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.annotate("Mayor $\\sigma$ → mayor RMSE\n(ruido degrada la inferencia)",
                xy=(0.05, 0.95), xycoords="axes fraction", fontsize=7.5, va="top", color="gray")

fig4.suptitle("1d) Efecto del ruido en la inferencia de (\\gamma, k)\n"
             "\\gamma es más difícil de inferir (RMSE_\\gamma > RMSE_k escalado) "
             "porque su señal es la envolvente lenta, vulnerable al ruido", fontsize=10)
plt.tight_layout()

# Guardado estratégico y despliegue del Gráfico 1D (Estudio del ruido)
path4 = os.path.join(output_dir, "1d_efecto_ruido_rmse.png")
plt.savefig(path4, dpi=150)
display(fig4)
plt.close(fig4)


# ──────────────────────────────────────────────────────────────────────────────
# RESUMEN NUMÉRICO
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("RESUMEN FINAL – \\sigma = 0.02 (datos base)")
print(f"  Random Forest: RMSE_\\gamma={rmse_rf_gamma:.4f}, RMSE_k={rmse_rf_k:.4f}")
print(f"  MLP PyTorch:   RMSE_\\gamma={rmse_mlp_gamma:.4f}, RMSE_k={rmse_mlp_k:.4f}")
print("="*55)

<Figure size 1300x700 with 6 Axes>


Entrenando RandomForest...
  RF  → RMSE_\gamma = 0.0152,  RMSE_k = 0.0159
  MLP → RMSE_\gamma = 0.0138,  RMSE_k = 0.0411


<Figure size 1100x900 with 4 Axes>

<Figure size 700x400 with 1 Axes>


── \sigma = 0.0 ──────────────────────────────────
  RF  → RMSE_\gamma=0.0093, RMSE_k=0.0095
  MLP → RMSE_\gamma=0.0179, RMSE_k=0.0623

── \sigma = 0.01 ──────────────────────────────────
  RF  → RMSE_\gamma=0.0126, RMSE_k=0.0135
  MLP → RMSE_\gamma=0.0254, RMSE_k=0.0866

── \sigma = 0.02 ──────────────────────────────────
  RF  → RMSE_\gamma=0.0170, RMSE_k=0.0248
  MLP → RMSE_\gamma=0.0175, RMSE_k=0.0830

── \sigma = 0.05 ──────────────────────────────────
  RF  → RMSE_\gamma=0.0322, RMSE_k=0.0519
  MLP → RMSE_\gamma=0.0241, RMSE_k=0.1080

── \sigma = 0.1 ──────────────────────────────────
  RF  → RMSE_\gamma=0.0601, RMSE_k=0.0855
  MLP → RMSE_\gamma=0.0335, RMSE_k=0.1031


<Figure size 1200x500 with 2 Axes>


RESUMEN FINAL – \sigma = 0.02 (datos base)
  Random Forest: RMSE_\gamma=0.0152, RMSE_k=0.0159
  MLP PyTorch:   RMSE_\gamma=0.0138, RMSE_k=0.0411


In [17]:
"""
Problema 2
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import quad
import warnings
import os  
from IPython.display import display, Image  

warnings.filterwarnings("ignore")


output_dir = "outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

#CONSTANTES FÍSICAS 
M_P      = 938.272   #masa protón
M_E      = 0.511     #masa e-
R_E      = 2.818e-13 #radio clásico del electrón
K_BB     = 0.307075  #constante de Bethe-Bloch

#H2O
Z_W      = 10.0      # número atómico ef
A_W      = 18.0      # masa atómica 
I_W      = 75e-6     
RHO_W    = 1.0       #densidad
X0_W     = 36.08     #longitud de radiación del agua
Ne_W     = RHO_W * 6.022e23 * Z_W / A_W  #(e-)/cm³

Z_P      = 1


#B) BETHE-BLOCH Y RANGO CSDA

def beta_gamma_from_E(E_kin, M=M_P):
    """calculamos el beta y el gamma"""
    gamma_L = 1.0 + E_kin / M
    beta    = np.sqrt(1.0 - 1.0 / gamma_L**2)
    return beta, gamma_L


def T_max(beta, gamma_L):
    """Máxima energía transferible a un e⁻ libre, en MeV"""
    return (2.0 * M_E * beta**2 * gamma_L**2 /
            (1.0 + 2.0 * gamma_L * M_E / M_P + (M_E / M_P)**2))


def bethe_bloch(E_kin):
    """
    Pérdida de energía por unidad de longitud de masa: -(dE/dx) [MeV cm²/g].
    Se multiplica por rho para obtener [MeV/cm].
    Válido para protones en agua.
    """
    if E_kin <= 0:
        return np.inf
    beta, gamma_L = beta_gamma_from_E(E_kin)
    if beta < 1e-6:
        return np.inf

    Tmax = T_max(beta, gamma_L)
    log_arg = (2.0 * M_E * beta**2 * gamma_L**2 * Tmax) / I_W**2

    dEdx_mass = (K_BB * Z_P**2 * (Z_W / A_W) / beta**2 *
                 (0.5 * np.log(log_arg) - beta**2))  

    return max(dEdx_mass, 1e-6)   #cota inferior para evitar divergencias


def dEdx_cm(E_kin):
    """Pérdida de energía por unidad de longitud [MeV/cm] en agua."""
    return bethe_bloch(E_kin) * RHO_W


def csda_range(E0_MeV):
    """
    Rango CSDA [cm]: integral \\int₀^{E₀} dE / (dE/dx)(E).
    Se integra desde E_min=0.01 MeV para evitar la singularidad en E=0.
    """
    E_min = 0.01  # MeV — límite inferior (protón casi en reposo)
    result, _ = quad(lambda E: 1.0 / dEdx_cm(E),
                     E_min, E0_MeV,
                     limit=200, epsabs=1e-6, epsrel=1e-6)
    return result  # cm


#calculamos los rangos CSDA para E0 = 50, 150, 250 MeV 
print("=" * 55)
print("2b) Rangos CSDA calculados (Bethe-Bloch numérico)")
print("=" * 55)

energies   = [50, 150, 250]
# Valores tabulados PSTAR/NIST para protones en agua [cm]
pstar_vals = {50: 2.18, 150: 15.8, 250: 38.0}

csda_vals = {}
for E0 in energies:
    R = csda_range(E0)
    csda_vals[E0] = R
    pstar = pstar_vals[E0]
    err   = abs(R - pstar) / pstar * 100
    print(f"  E0 = {E0:3d} MeV:  RCSDA = {R:.2f} cm  "
          f"| PSTAR = {pstar:.2f} cm  | error = {err:.1f}%")
print()

# graficamos dE/dx vs energía
E_arr   = np.logspace(-1, 3, 500)          # 0.1 – 1000 MeV
dEdx_arr = np.array([dEdx_cm(E) for E in E_arr])

fig1, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.loglog(E_arr, dEdx_arr, "b-", lw=2)
ax.set_xlabel("Energía cinética $E$ [MeV]", fontsize=11)
ax.set_ylabel(r"$-dE/dx$ [MeV/cm]", fontsize=11)
ax.set_title("2b) Bethe-Bloch para protones en agua", fontsize=11)
ax.grid(True, which="both", alpha=0.3)
for E0 in energies:
    ax.axvline(E0, ls="--", color="red", lw=0.8, alpha=0.6)
    ax.text(E0 * 1.05, 2, f"{E0} MeV", color="red", fontsize=8)

ax = axes[1]
E_range = np.linspace(0.1, 260, 600)
R_arr   = [csda_range(E) for E in E_range]
ax.plot(E_range, R_arr, "b-", lw=2, label="CSDA calculado")
for E0 in energies:
    ax.scatter([E0], [csda_vals[E0]], color="blue", s=80, zorder=5)
    ax.scatter([E0], [pstar_vals[E0]], color="red",  s=80,
               marker="*", zorder=5, label=f"PSTAR {E0} MeV" if E0 == 50 else "")
    ax.annotate(f"  {E0} MeV\n  calc={csda_vals[E0]:.1f} cm\n  NIST={pstar_vals[E0]:.1f} cm",
                (E0, csda_vals[E0]), fontsize=7.5, color="darkblue")
ax.set_xlabel("$E_0$ [MeV]", fontsize=11)
ax.set_ylabel("$R_{CSDA}$ [cm]", fontsize=11)
ax.set_title("2b) Rango CSDA vs energía inicial", fontsize=11)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig1.suptitle("2b) Bethe-Bloch y CSDA — protones en agua ($I=75$ eV)", fontsize=12)
plt.tight_layout()

path1 = os.path.join(output_dir, "2b_bethe_bloch_csda.png")
plt.savefig(path1, dpi=150)
display(fig1)
plt.close(fig1)


#C) MONTE CARLO DETERMINISTA (sin straggling)

E0_MC  = 150.0          
dx_cm  = 0.01           
N_P    = 10_000         #número de protones

R_csda_150 = csda_vals[150]
print(f"2c) CSDA E0=150 MeV: {R_csda_150:.2f} cm")

E_table   = np.linspace(0.01, 160.0, 4000)
dEdx_table = np.array([dEdx_cm(E) for E in E_table])

def dEdx_interp(E):
    """Interpolación lineal de la tabla dE/dx."""
    if E <= E_table[0]:  return dEdx_table[0]
    if E >= E_table[-1]: return dEdx_table[-1]
    return np.interp(E, E_table, dEdx_table)


def simulate_deterministic(N, E0, dx):
    """
    Transporta N protones paso a paso sin fluctuación. Devuelve histograma de dosis D(z).
    """
    z_max   = int(R_csda_150 * 1.3 / dx) + 1
    dose    = np.zeros(z_max)

    for _ in range(N):
        E = E0
        iz = 0
        while E > 0.01 and iz < z_max - 1:
            dE = dEdx_interp(E) * dx
            dE = min(dE, E)
            dose[iz] += dE
            E  -= dE
            iz += 1
        if iz < z_max:
            dose[iz] += E   #depositar energía remanente

    z_arr = np.arange(z_max) * dx * 10  #convertimos a mm
    return z_arr, dose / N              #dosis media por protón


print(f"  Simulando {N_P} protones deterministas...")
z_det, D_det = simulate_deterministic(N_P, E0_MC, dx_cm)

# Pico de Bragg determinista
iz_peak_det = np.argmax(D_det)
z_peak_det  = z_det[iz_peak_det]
print(f"  Pico de Bragg (sin straggling): z = {z_peak_det:.1f} mm "
      f"= {z_peak_det/10:.2f} cm  (RCSDA = {R_csda_150:.2f} cm)")


#D) MONTE CARLO CON STRAGGLING 

def sigma_E_bohr(E_kin, dx):
    """
    Desviación estándar de la fluctuación energética por paso. Aproximación gaussiana de Bohr: sigma_E^{2} = 4pi r_e^{2} (m_e c^{2})^{2}N_e (z^{2}/beta^{2}) Deltax
    """
    beta, _ = beta_gamma_from_E(E_kin)
    if beta < 1e-6: return 0.0
    sigma2 = (4.0 * np.pi * R_E**2 * M_E**2 * Ne_W *
              Z_P**2 / beta**2 * dx)   # MeV²
    return np.sqrt(max(sigma2, 0.0))


def simulate_straggling(N, E0, dx, rng=None):
    """
    Transporta N protones con fluctuación gaussiana en cada paso. Devuelve histograma de dosis D(z).
    """
    if rng is None:
        rng = np.random.default_rng()

    z_max = int(R_csda_150 * 1.5 / dx) + 1
    dose  = np.zeros(z_max)

    for _ in range(N):
        E  = E0
        iz = 0
        while E > 0.01 and iz < z_max - 1:
            dE_mean = dEdx_interp(E) * dx
            sigma   = sigma_E_bohr(E, dx)
            dE      = dE_mean + rng.normal(0.0, sigma)
            dE      = np.clip(dE, 0.0, E)       # no puede ganar energía
            dose[iz] += dE
            E  -= dE
            iz += 1
        if iz < z_max:
            dose[iz] += E

    z_arr = np.arange(z_max) * dx * 10   # mm
    return z_arr, dose / N


print(f"  Simulando {N_P} protones con straggling...")
rng = np.random.default_rng(88)
z_str, D_str = simulate_straggling(N_P, E0_MC, dx_cm, rng)

# Pico de Bragg con straggling
iz_peak_str = np.argmax(D_str)
z_peak_str  = z_str[iz_peak_str]

#medimos ensanchamiento
def fwhm_peak(z, D):
    """Calcula FWHM del pico de Bragg usando el máximo y umbral al 50%."""
    i_pk   = np.argmax(D)
    D_half = D[i_pk] * 0.5
    # lado izquierdo
    left   = z[:i_pk][D[:i_pk] >= D_half]
    right  = z[i_pk:][D[i_pk:] >= D_half]
    if len(left) == 0 or len(right) == 0:
        return np.nan
    fwhm = right[-1] - left[0]
    return fwhm

fwhm_det = fwhm_peak(z_det, D_det)
fwhm_str = fwhm_peak(z_str, D_str)
sigma_R  = fwhm_str / (2 * np.sqrt(2 * np.log(2)))  # FWHM → \\sigma gaussiano

print(f"  Pico (con straggling):     z = {z_peak_str:.1f} mm")
print(f"  FWHM sin straggling:       {fwhm_det:.1f} mm")
print(f"  FWHM con straggling:       {fwhm_str:.1f} mm")
print(f"  Ensanchamiento \\sigma_R:         {sigma_R:.2f} mm")


#GRÁFICOS 2C y 2D

fig2, axes = plt.subplots(1, 2, figsize=(14, 6))

#panel izquierdo: comparación de perfiles de dosis
ax = axes[0]
ax.plot(z_det, D_det, "b-",  lw=2,   label="Sin straggling (determinista)")
ax.plot(z_str, D_str, "r--", lw=2,   label="Con straggling (Bohr gaussiano)")
ax.axvline(z_peak_det,  color="blue",  ls=":",  lw=1.5,
           label=f"Pico det. z={z_peak_det:.1f} mm")
ax.axvline(z_peak_str,  color="red",   ls=":",  lw=1.5,
           label=f"Pico str. z={z_peak_str:.1f} mm")
ax.axvline(R_csda_150 * 10, color="green", ls="-.", lw=1.5,
           label=f"RCSDA = {R_csda_150:.2f} cm = {R_csda_150*10:.1f} mm")
ax.set_xlabel("Profundidad z [mm]", fontsize=11)
ax.set_ylabel("Dosis media por protón [MeV]", fontsize=11)
ax.set_title(f"2c–2d) Perfil de dosis D(z)\n$E_0 = {E0_MC}$ MeV, "
             f"N = {N_P:,} protones, $\\Delta x = {dx_cm*10:.1f}$ mm",
             fontsize=10)
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.3)

# paanel derecho: zoom sobre el pico de Bragg 
ax = axes[1]
z_lo = z_peak_str - 25
z_hi = z_peak_str + 20
mask_det = (z_det >= z_lo) & (z_det <= z_hi)
mask_str = (z_str >= z_lo) & (z_str <= z_hi)

ax.plot(z_det[mask_det], D_det[mask_det], "b-",  lw=2.5, label="Sin straggling")
ax.plot(z_str[mask_str], D_str[mask_str], "r--", lw=2.5, label="Con straggling")
ax.set_xlabel("Profundidad z [mm]", fontsize=11)
ax.set_ylabel("Dosis media por protón [MeV]", fontsize=11)
ax.set_title(f"2d) Zoom pico de Bragg\n"
             f"FWHM det = {fwhm_det:.1f} mm  |  FWHM str = {fwhm_str:.1f} mm  "
             f"|  $\\sigma_R$ = {sigma_R:.2f} mm",
             fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

for D_arr, z_arr, color, ls in [(D_det, z_det, "blue", "-"),
                                  (D_str, z_str, "red",  "--")]:
    i_pk   = np.argmax(D_arr)
    D_half = D_arr[i_pk] * 0.5
    left   = z_arr[:i_pk][D_arr[:i_pk] >= D_half]
    right  = z_arr[i_pk:][D_arr[i_pk:] >= D_half]
    if len(left) > 0 and len(right) > 0:
        ax.annotate("", xy=(right[-1], D_half), xytext=(left[0], D_half),
                    arrowprops=dict(arrowstyle="<->", color=color, lw=1.5))

fig2.suptitle("2c–2d) Simulación Monte Carlo de protones en agua\n"
             r"Bethe-Bloch + straggling de Bohr ($E_0=150$ MeV)", fontsize=12)
plt.tight_layout()

path2 = os.path.join(output_dir, "2cd_dosis_bragg.png")
plt.savefig(path2, dpi=150)
display(fig2)
plt.close(fig2)


#graficamos la dosis normalizada
fig3, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_det, D_det / D_det.max() * 100, "b-",  lw=2,
        label="Sin straggling")
ax.plot(z_str, D_str / D_str.max() * 100, "r--", lw=2,
        label="Con straggling")
ax.set_xlabel("Profundidad z [mm]", fontsize=11)
ax.set_ylabel("Dosis normalizada [%]", fontsize=11)
ax.set_title(f"2d) Perfil de dosis normalizado — $E_0={E0_MC}$ MeV\n"
             "El straggling ensancha y desplaza levemente el pico de Bragg", fontsize=10)
ax.axhline(50, color="gray", ls=":", lw=1.0)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()

path3 = os.path.join(output_dir, "2d_dosis_normalizada.png")
plt.savefig(path3, dpi=150)
display(fig3)
plt.close(fig3)


#resumen
print()
print("=" * 55)
print("RESUMEN FINAL — Problema 2")
print("=" * 55)
print(f"  CSDA (50  MeV): {csda_vals[50]:.2f} cm  (PSTAR: 2.18 cm)")
print(f"  CSDA (150 MeV): {csda_vals[150]:.2f} cm  (PSTAR: 15.8 cm)")
print(f"  CSDA (250 MeV): {csda_vals[250]:.2f} cm  (PSTAR: 38.0 cm)")
print(f"  Pico Bragg det.:    {z_peak_det:.1f} mm = {z_peak_det/10:.2f} cm")
print(f"  Pico Bragg strag.:  {z_peak_str:.1f} mm = {z_peak_str/10:.2f} cm")
print(f"  FWHM sin straggling:  {fwhm_det:.1f} mm")
print(f"  FWHM con straggling:  {fwhm_str:.1f} mm")
print(f"  \\sigma_R (Gauss):          {sigma_R:.2f} mm")
print("=" * 55)

2b) Rangos CSDA calculados (Bethe-Bloch numérico)
  E0 =  50 MeV:  RCSDA = 2.22 cm  | PSTAR = 2.18 cm  | error = 1.8%
  E0 = 150 MeV:  RCSDA = 15.75 cm  | PSTAR = 15.80 cm  | error = 0.3%
  E0 = 250 MeV:  RCSDA = 37.90 cm  | PSTAR = 38.00 cm  | error = 0.3%



<Figure size 1300x500 with 2 Axes>

2c) CSDA E0=150 MeV: 15.75 cm
  Simulando 10000 protones deterministas...
  Pico de Bragg (sin straggling): z = 157.6 mm = 15.76 cm  (RCSDA = 15.75 cm)
  Simulando 10000 protones con straggling...
  Pico (con straggling):     z = 145.6 mm
  FWHM sin straggling:       0.4 mm
  FWHM con straggling:       16.8 mm
  Ensanchamiento \sigma_R:         7.13 mm


<Figure size 1400x600 with 2 Axes>

<Figure size 1000x500 with 1 Axes>


RESUMEN FINAL — Problema 2
  CSDA (50  MeV): 2.22 cm  (PSTAR: 2.18 cm)
  CSDA (150 MeV): 15.75 cm  (PSTAR: 15.8 cm)
  CSDA (250 MeV): 37.90 cm  (PSTAR: 38.0 cm)
  Pico Bragg det.:    157.6 mm = 15.76 cm
  Pico Bragg strag.:  145.6 mm = 14.56 cm
  FWHM sin straggling:  0.4 mm
  FWHM con straggling:  16.8 mm
  \sigma_R (Gauss):          7.13 mm
